In [ ]:
#Imports
import numpy as np
from kinematics_functions import *


# Spesific problem parameters

# d values, consisten with following answer to question 1, not consistent with estimates we did for kinematics part of poject
d_1 = 0.34
d_3 = 0.40    # upper arm  (shoulder → elbow)
d_5 = 0.40    # forearm    (elbow → wrist)
d_7 = 0.126   # flange/tool (wrist → flange)

# DH_table
def DH(q):
    return np.array([
        # d,    theta,  a,  alpha
        [d_1,   q[0],   0,  np.pi/2 ],
        [0,     q[1],   0, -np.pi/2 ],
        [d_3,   q[2],   0, -np.pi/2 ],
        [0,     q[3],   0,  np.pi/2 ],
        [d_5,   q[4],   0,  np.pi/2 ],
        [0,     q[5],   0, -np.pi/2 ],
        [d_7,   q[6],   0,  0       ],
    ])


# Question 1 - Approximate Link Inertias 

## Method

Each of the 7 links is approximated as a **single solid cylinder** sized to the
arm's outer envelope. Lengths come from the published iiwa kinematic offsets
(0.34 / 0.40 / 0.40 / 0.126 m for the limb segments; the shoulder, elbow and
wrist joint-pairs are co-located, so their housings are modelled as short
cylinders). Radii are read off the photograph, tapering from ~65 mm at the base
to ~35 mm at the flange.

A real cobot link is a hollow aluminium shell containing a motor, gearbox and
torque sensor. Instead of modelling those internals, the solid cylinder is given
a single **effective density** ρ = 1565 kg/m³ (~58% of solid aluminium). That
value is calibrated so the seven cylinder masses sum to the catalogue robot mass
of ~24 kg. Using the larger cylinders for the proximal links automatically puts
more mass near the base, as in the real arm.

Link frame *i* sits at joint *i* with its **z-axis along the joint axis**. Since
each cylinder is axisymmetric about that axis, every COM lies on z and each
inertia tensor is **diagonal** in the link frame (products of inertia ≈ 0).

## Formulas

For a cylinder of mass m = ρ·πr²L, length L along its axis, radius r:

- axial inertia: **Izz = ½ m r²**
- transverse inertia: **Ixx = Iyy = (1/12) m (3r² + L²)**

both taken about the COM, located at the cylinder's mid-length.

## Per-link results

| Link | Calculation (L, r → m; Izz; Ixx=Iyy) |
|---|---|
| 1 base/shoulder | L=0.34, r=0.065 → m=7.06 kg; Izz=0.0149; Ixx=Iyy=0.0755; com_z=0.150 |
| 2 shoulder pitch | L=0.18, r=0.060 → m=3.19 kg; Izz=0.00573; Ixx=Iyy=0.0115; com_z=0 |
| 3 upper arm | L=0.40, r=0.055 → m=5.95 kg; Izz=0.00900; Ixx=Iyy=0.0838; com_z=0.180 |
| 4 elbow pitch | L=0.16, r=0.050 → m=1.97 kg; Izz=0.00246; Ixx=Iyy=0.00542; com_z=0 |
| 5 forearm | L=0.40, r=0.045 → m=3.98 kg; Izz=0.00403; Ixx=Iyy=0.0551; com_z=0.170 |
| 6 wrist pitch | L=0.14, r=0.040 → m=1.10 kg; Izz=0.000881; Ixx=Iyy=0.00224; com_z=0 |
| 7 flange/tool | L=0.126, r=0.035 → m=0.76 kg; Izz=0.000465; Ixx=Iyy=0.00124; com_z=0.050 |

Units: kg, m, kg·m². Total mass = 24.0 kg.

## Python representation

In [2]:
# KUKA LBR Med 7 R800 — approximate inertial parameters of the 7 links.
# Each link modelled as a solid cylinder; effective density 1565 kg/m^3
# calibrated so masses sum to ~24 kg. Inertia about COM, in link frame.
# Row = [mass, cx, cy, cz, Ixx, Iyy, Izz, Ixy, Ixz, Iyz]   (units: kg, m, kg*m^2)
LINKS = np.array([
    [7.061, 0, 0, 0.150, 7.5475e-2, 7.5475e-2, 1.4916e-2, 0, 0, 0],
    [3.185, 0, 0, 0.000, 1.1466e-2, 1.1466e-2, 5.7330e-3, 0, 0, 0],
    [5.947, 0, 0, 0.180, 8.3796e-2, 8.3796e-2, 8.9954e-3, 0, 0, 0],
    [1.966, 0, 0, 0.000, 5.4231e-3, 5.4231e-3, 2.4576e-3, 0, 0, 0],
    [3.981, 0, 0, 0.170, 5.5099e-2, 5.5099e-2, 4.0310e-3, 0, 0, 0],
    [1.101, 0, 0, 0.000, 2.2387e-3, 2.2387e-3, 8.8080e-4, 0, 0, 0],
    [0.759, 0, 0, 0.050, 1.2360e-3, 1.2360e-3, 4.6468e-4, 0, 0, 0],
])
 
mass = LINKS[:, 0]
com  = LINKS[:, 1:4]
# inertia tensor per link (diagonal here, off-diagonals ~0):
inertia = np.array([np.diag(r[4:7]) for r in LINKS])    

# Question 2 - Making the model

We need to make a model. This basically means that we need to make some sort of code where we can test out different functions for calculating the joint torque vector, then we need to be able to calculate the resulting motion of the whole robot arm and plot it and evaluate it.

## Framework
Fundamentally, this model will be based on a function that takes in the current state, current timestep, and some sort of function for that returns the torque vector, and then returns the velocity and acelleration, like this template function:
```
def dynamics(t, q, q_dot, tau_func):
    tau = tau_func(t, q, q_dot)
    q_ddot = forward_dynamics(q, q_dot, tau)
    return q_dot, q_ddot
```
This function can then be solved with an ODE solver for example from scipy to get the full behaviour of the robot arm. Saving an array with the states at the different timesteps will allow us to easily plot the movement of the robot arm.


## Implementation of the forward_dynamics function
Now that we have a plan for the framework, let's start diving into the requirements for implementing the function above. The tau function will be based on what control algorithm is used, so this will be dealt with in question 3 and 4, for now we can just ignore it. The forward_dynamics function we need to implement now. This function will be based on the equation:
$$ \Tau = B(q)\ddot{q} + C(q, \dot{q})\dot{q} + g(q) $$
This equation can easily be rewritten so that $\ddot{q}$ stands alone, giving us exactly what we need for the forward_dynamics function. However, we need to find the expressions for $M(q)$, $C(q, \dot{q})$ and $g(q)$. LEt's find these terms one by one.

### Inertia matrix
The mass and inertia matrix $B(q)$ can be found either by using a Lagrangian approach, or by using the newton-euler method. We will now find it using the newton-euler method because it is slightly more efficient, and because it is harder to implement so we will learn more by implementing it. 

Row n in B(q) represents all the torques required to accelerate joint n. When we use the Newton-Euler approach, we find each of these columns by doing a forward and backwards pass. We start by assuming that all velocities and gravity are zero. Then, in the forwards pass, we calculate how each link will move when we have an acceleration on joint n. To calculate this, we use formulas based on Newtons equations. For a revolute joint, the forward Newton--Euler pass can be written as:

$$\boldsymbol{\omega}_i={}^{i}\!R_{i-1}\boldsymbol{\omega}_{i-1}+\mathbf{z}_i \dot q_i$$

$$\dot{\boldsymbol{\omega}}_i={}^{i}\!R_{i-1}\dot{\boldsymbol{\omega}}_{i-1}+\mathbf{z}_i \ddot q_i+\left({}^{i}\!R_{i-1}\boldsymbol{\omega}_{i-1}\right)\times\left(\mathbf{z}_i \dot q_i\right)$$

$$\mathbf{a}_i={}^{i}\!R_{i-1}\left(\mathbf{a}_{i-1}+\dot{\boldsymbol{\omega}}_{i-1} \times \mathbf{p}_i+\boldsymbol{\omega}_{i-1}\times\left(\boldsymbol{\omega}_{i-1} \times \mathbf{p}_i\right)\right)$$

$$\mathbf{a}_{c_i}=\mathbf{a}_i+\dot{\boldsymbol{\omega}}_i \times \mathbf{r}_{c_i}+\boldsymbol{\omega}_i\times\left(\boldsymbol{\omega}_i \times \mathbf{r}_{c_i}\right)$$

Then, in the following backwards pass we use another set of formulas based on newtons laws to calculate the torque felt in each joint based on the accelerations of the link that comes after the joint and the force that is transmitted from the next joint. {Claude, help me out and insert formulas here}

Below is an implemented frunction B(q) that is based on this method.

In [ ]:
# CLAUDE, PUT IMPLEMENTATION OF B(q) USING NEWTON-EULER METHOD HERE